# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、学習前後を比べる

講座ノートブック `lora_format_tuning.ipynb` の環境構築・学習設定を流用し、FAQ の代わりに `scripts/make_sft_dataset.py` で作った用語集の Q&A（116 件）で学習します。評価用の質問は自作です。


## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 5,   # 発展: 学習データ 116 件 × 5 周（1 周 15 ステップ、合計 75 ステップ）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,   # 発展: 用語集の定義（最長 212 文字）が途中で切れないように増やす
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/glossary_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 用語集の SFT データと評価用の質問


In [ ]:
import urllib.request

# scripts/make_sft_dataset.py で作った JSONL（自分の fork の lec02 ブランチに置いたもの）を読み込む
SFT_URL = "https://raw.githubusercontent.com/AtsunoriNagaya/lecture-ai-engineering-2026/lec02/lectures/02/glossary_sft.jsonl"
sft_rows = [json.loads(l) for l in urllib.request.urlopen(SFT_URL).read().decode("utf-8").splitlines() if l.strip()]
print("学習データ:", len(sft_rows), "件 | 1 件目:", sft_rows[0]["messages"][0]["content"])

# 評価用の質問と正解（自作）。種類は 3 つ
#   as_is      : 用語集にある語を、学習データと同じ言い回しで聞く
#   paraphrase : 用語集にある語を、学習データに無い言い回しで聞く
#   unknown    : 用語集に無い語を聞く（正解は「用語集に無い／分からない」と答えること）
EVAL = [
    ("as_is", "「LoRA」とは何ですか？", ["低ランク"]),
    ("as_is", "「既定モデル」とは何ですか？", ["gemini-3.5-flash-lite"]),
    ("as_is", "「宿題の3レベル」とは何ですか？", ["基礎", "実装", "発展"]),
    ("as_is", "「MCPサーバー」とは何ですか？", ["公開"]),
    ("paraphrase", "Low-Rank Adaptation ってどんな仕組みですか？", ["低ランク"]),
    ("paraphrase", "この講座の演習で標準的に使うモデルの名前を教えてください。", ["gemini-3.5-flash-lite"]),
    ("paraphrase", "AIE2026 の宿題にはどんなレベルがありますか？", ["基礎", "実装", "発展"]),
    ("paraphrase", "モデルの重みを 4 ビットで持つと何がうれしいのですか？", ["メモリ"]),
    ("unknown", "「RAG」とは何ですか？", None),
    ("unknown", "「蒸留」について簡単に説明してください。", None),
    ("unknown", "「lec07-final タグ」とは何ですか？", None),
    ("unknown", "AIE2026 の修了証はいつ発行されますか？", None),
]
# unknown の質問で、この語句のどれかを含めば「保留（分からないと答えた）」、含まなければ「断定」とみなす
HEDGE_WORDS = ["用語集", "見つかりません", "わかりません", "分かりません", "記載がありません", "確認", "不明", "存在しません"]


## 3. モデルの読み込みと学習前評価


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
pd.set_option("display.max_colwidth", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE)

def generate(m, question: str) -> str:
    # 学習データと同じく、system なし・user 1 発話だけで聞く
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def judge(kind, raw, must):
    if kind == "unknown":
        return "保留" if any(w in raw for w in HEDGE_WORDS) else "断定"
    return "○" if all(w.lower() in raw.lower() for w in must) else "×"

def evaluate(m, label):
    rows = []
    for kind, q, must in EVAL:
        raw = generate(m, q)
        rows.append({"kind": kind, "question": q, "raw": raw, "judge": judge(kind, raw, must)})
    (WORK / f"eval_{label}.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return rows

def summarize(rows):
    s = {}
    for kind in ["as_is", "paraphrase"]:
        rs = [r for r in rows if r["kind"] == kind]
        s[f"{kind}_correct"] = f'{sum(r["judge"] == "○" for r in rs)}/{len(rs)}'
    rs = [r for r in rows if r["kind"] == "unknown"]
    s["unknown_hedged"] = f'{sum(r["judge"] == "保留" for r in rs)}/{len(rs)}'
    return s

before_rows = evaluate(model, "before")
print(json.dumps(summarize(before_rows), ensure_ascii=False))
display(pd.DataFrame(before_rows))


## 4. LoRA 学習


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(sft_rows)   # 各行が messages 形式なので、そのまま渡せる
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=5, save_steps=50, save_total_limit=1,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
FINAL_LOSS = [h["loss"] for h in trainer.state.log_history if "loss" in h][-1]
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | 最終 loss: {FINAL_LOSS}")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()


## 5. 学習後評価と result.json


In [ ]:
after_rows = evaluate(tuned, "after")
print("学習前:", json.dumps(summarize(before_rows), ensure_ascii=False))
print("学習後:", json.dumps(summarize(after_rows), ensure_ascii=False))
display(pd.DataFrame([{"種類": b["kind"], "質問": b["question"], "学習前": b["judge"], "学習前の出力": b["raw"],
                       "学習後": a["judge"], "学習後の出力": a["raw"]} for b, a in zip(before_rows, after_rows)]))

result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）・用語集 SFT",
    "train_records": len(sft_rows), "train_seconds": TRAIN_SECONDS, "final_loss": FINAL_LOSS,
    "before": summarize(before_rows), "after": summarize(after_rows),
    "rows": [{"kind": b["kind"], "question": b["question"], "before": b["judge"], "before_raw": b["raw"],
              "after": a["judge"], "after_raw": a["raw"]} for b, a in zip(before_rows, after_rows)],
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("保存しました:", WORK / "result.json")
